# TP6 — Attaques Adversariales & MITRE ATLAS
**Module 3 — Deep Learning · M1 IA & Cybersécurité Aéronautique & Spatial**

---

> 🛩️ **Contexte** : Vous êtes red team dans une cellule de sécurité aéronautique.  
> Le LSTM-IDS déployé sur les logs ACARS (TP5) vient d'être validé en production.  
> Votre mission : **tenter de le tromper**, documenter l'attaque dans MITRE ATLAS,  
> puis proposer une défense.

---

## Objectifs
1. Comprendre ce qu'est une attaque adversariale sur un modèle séquentiel
2. Implémenter FGSM (Fast Gradient Sign Method) pas à pas
3. Simuler un empoisonnement de dataset (data poisoning)
4. Mapper ces attaques sur la matrice **MITRE ATLAS**
5. Appliquer une défense simple (adversarial training)

## Prérequis
```
pip install torch numpy pandas matplotlib scikit-learn
```

---
## Partie 0 — Rappel : MITRE ATLAS

**MITRE ATT&CK** catalogue les attaques contre les systèmes informatiques classiques.  
**MITRE ATLAS** (*Adversarial Threat Landscape for AI Systems*) fait la même chose pour les **systèmes IA**.

| Tactique ATLAS | Ce que ça veut dire | Exemple dans ce TP |
|---|---|---|
| Reconnaissance | Comprendre le modèle cible | Observer les sorties du LSTM |
| ML Attack Staging | Préparer l'attaque | Générer des exemples FGSM |
| Exfiltration | Voler des informations sur le modèle | Model stealing (hors scope ici) |
| Impact | Faire prendre une mauvaise décision | Faire classer une attaque en trafic normal |
| Persistence | Maintenir un accès/effet | Backdoor dans les données d'entraînement |

> 📌 Référence officielle : https://atlas.mitre.org  
> À chaque étape du TP, vous allez **remplir une fiche ATLAS** pour documenter l'attaque réalisée.

---
## Partie 1 — Mise en place : le modèle cible

On repart du LSTM-IDS simplifié du TP5.  
Il prend en entrée une séquence de 10 logs ACARS et prédit : **0 = normal, 1 = attaque**.

### 🛩️ Analogie
Imaginez un copilote automatique qui surveille le trafic radio.  
On va lui faire entendre des messages **subtilement** altérés pour qu'il pense que tout va bien — alors qu'une attaque est en cours.  
La clé : la modification doit être **imperceptible** à l'œil humain.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports OK")
print(f"PyTorch version : {torch.__version__}")

In [ ]:
# ── Génération du dataset ACARS simulé ──────────────────────────────────────
# Chaque log est un vecteur de 6 features :
# [type_msg_encoded, lat_norm, lon_norm, alt_norm, vitesse_norm, source_connue]
#
# IMPORTANT — deux types d'attaques dans le dataset :
#   • Attaque franche  (label=1, score ~0.95) : anomalie évidente, saut brutal
#   • Attaque subtile  (label=1, score ~0.65) : légère déviation progressive
#     → ce sont ces dernières que FGSM va faire passer en dessous du seuil

def generate_acars_dataset(n_sequences=2000, seq_len=10, n_features=6):
    X, y, attack_type = [], [], []

    for _ in range(n_sequences):
        r = np.random.random()

        if r < 0.5:
            # ── Séquence normale ──────────────────────────────────────────
            seq = np.random.normal(
                loc=[0.2, 0.85, 0.05, 0.6, 0.5, 1.0],
                scale=[0.05, 0.02, 0.02, 0.05, 0.05, 0.0],
                size=(seq_len, n_features)
            )
            y.append(0)
            attack_type.append('normal')

        elif r < 0.75:
            # ── Attaque franche : saut brutal (score > 0.9 facilement) ───
            seq = np.random.normal(
                loc=[0.2, 0.85, 0.05, 0.6, 0.5, 1.0],
                scale=[0.05, 0.02, 0.02, 0.05, 0.05, 0.0],
                size=(seq_len, n_features)
            )
            attack_start = np.random.randint(4, 7)
            seq[attack_start:, 0] = 0.9   # type CMD inhabituel
            seq[attack_start:, 1] = 0.0   # lat = 0.00
            seq[attack_start:, 2] = 0.0   # lon = 0.00
            seq[attack_start:, 5] = 0.0   # source inconnue
            y.append(1)
            attack_type.append('franche')

        else:
            # ── Attaque subtile : dérive progressive (score ~0.6–0.75) ───
            # Seules quelques features dévient légèrement sur les derniers pas
            seq = np.random.normal(
                loc=[0.2, 0.85, 0.05, 0.6, 0.5, 1.0],
                scale=[0.05, 0.02, 0.02, 0.05, 0.05, 0.0],
                size=(seq_len, n_features)
            )
            # Dérive progressive sur les 3 derniers pas seulement
            for t in range(7, 10):
                drift = (t - 6) * 0.08   # 0.08, 0.16, 0.24 — très faible
                seq[t, 0] += drift        # type_msg légèrement anormal
                seq[t, 5] -= drift        # source légèrement douteuse
            y.append(1)
            attack_type.append('subtile')

        X.append(np.clip(seq, 0, 1))

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        np.array(attack_type)
    )

X, y, attack_types = generate_acars_dataset()
X_train, X_test, y_train, y_test, at_train, at_test = train_test_split(
    X, y, attack_types, test_size=0.2, random_state=42
)

print(f"Dataset : {X.shape[0]} séquences | {X.shape[1]} pas de temps | {X.shape[2]} features")
print(f"Répartition : {(y==0).sum()} normales | {(attack_types=='franche').sum()} attaques franches | {(attack_types=='subtile').sum()} attaques subtiles")

In [ ]:
# ── Définition du LSTM-IDS ───────────────────────────────────────────────────

class LSTM_IDS(nn.Module):
    """
    LSTM-IDS : détecteur d'intrusion sur séquences ACARS.
    Architecture : 2 couches LSTM → couche dense → sigmoid
    Sortie : anomaly_score ∈ [0, 1]  (> 0.5 = alerte)
    """
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden  = lstm_out[:, -1, :]
        return self.classifier(last_hidden).squeeze(1)


model = LSTM_IDS()
print(model)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────────

def train_model(model, X_train, y_train, epochs=20, lr=1e-3, batch_size=64, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    dataset   = torch.utils.data.TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train)
    )
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        if verbose and (epoch+1) % 5 == 0:
            print(f"Époque {epoch+1:3d}/{epochs} | Loss : {avg:.4f}")
    return losses

print("Entraînement du LSTM-IDS...")
train_model(model, X_train, y_train, epochs=20)

model.eval()
with torch.no_grad():
    preds = model(torch.tensor(X_test))
    acc   = ((preds > 0.5).float() == torch.tensor(y_test)).float().mean()
print(f"\n✅ Précision globale : {acc*100:.1f}%")

# Précision par type d'attaque
for t in ['franche', 'subtile']:
    idx = np.where(at_test == t)[0]
    with torch.no_grad():
        s = model(torch.tensor(X_test[idx]))
    detected = (s > 0.5).float().mean()
    print(f"   Détection attaques {t:8s} : {detected*100:.1f}%  (score moyen : {s.mean():.2f})")

---
## Partie 2 — Attaque FGSM : tromper le modèle

### Principe

FGSM (*Fast Gradient Sign Method*, Goodfellow et al. 2014) exploite le gradient du modèle :

$$x_{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x \mathcal{L}(x, y_{cible}))$$

**Traduction** : on calcule dans quelle direction modifier chaque feature pour **maximiser l'erreur** du modèle. On fait un tout petit pas (epsilon) dans cette direction.

### ⚠️ Ce qu'on veut observer

Sur une **attaque subtile** — que le modèle détecte avec un score de 0.65 :  
→ Avant FGSM : score > 0.5 → **ALERTE déclenchée** ✅  
→ Après FGSM : score < 0.5 → **silence du modèle** ⚠️  

Et la modification des features est visuellement imperceptible.

### ATLAS — Technique ciblée
| Champ | Valeur |
|---|---|
| Tactique | **Impact** |
| Technique | **AML.T0015** — Evade ML Model |
| Sous-technique | Evasion via crafted inputs (white-box) |
| Cible | LSTM-IDS déployé sur logs ACARS |

In [ ]:
# ── Implémentation FGSM ──────────────────────────────────────────────────────

def fgsm_attack(model, x, y_true, epsilon):
    """
    Génère un exemple adversarial par FGSM.

    Paramètres
    ----------
    model   : le modèle à attaquer
    x       : tenseur d'entrée (1, seq_len, features)
    y_true  : label réel (0 ou 1)
    epsilon : amplitude de la perturbation (entre 0 et 1)

    Retourne
    --------
    x_adv : exemple perturbé — même forme que x
    """
    x_adv = x.clone().detach().requires_grad_(True)

    # Passe avant + calcul de la loss
    pred = model(x_adv)
    loss = nn.BCELoss()(pred, y_true)

    # Gradient par rapport à l'ENTRÉE (pas aux poids du modèle)
    model.zero_grad()
    loss.backward()

    # Petit pas dans la direction qui maximise l'erreur
    x_adv = x_adv + epsilon * x_adv.grad.sign()

    return torch.clamp(x_adv, 0, 1).detach()

In [ ]:
# ── Visualisation principale : score avant / après FGSM ──────────────────────
#
# On sélectionne 20 attaques SUBTILES correctement détectées par le modèle
# (score original > 0.5) et on applique FGSM avec différents epsilon.
# On montre comment le score bascule de « ALERTE » à « Normal ».

model.eval()
subtle_idx = np.where(at_test == 'subtile')[0]

# Garder uniquement ceux que le modèle détecte correctement AVANT perturbation
detected_subtle = []
for i in subtle_idx:
    x = torch.tensor(X_test[i:i+1])
    with torch.no_grad():
        s = model(x).item()
    if s > 0.5:
        detected_subtle.append((i, s))

print(f"Attaques subtiles correctement détectées : {len(detected_subtle)}")
print(f"Score moyen avant FGSM : {np.mean([s for _, s in detected_subtle]):.3f}")

# Pour chaque exemple, score original + scores après FGSM à différents epsilon
EPSILONS = [0.01, 0.03, 0.05, 0.10, 0.20]
examples = detected_subtle[:20]

scores_orig = [s for _, s in examples]
scores_adv  = {eps: [] for eps in EPSILONS}

for idx, _ in examples:
    x = torch.tensor(X_test[idx:idx+1])
    y = torch.tensor([y_test[idx]])
    for eps in EPSILONS:
        x_adv = fgsm_attack(model, x, y, epsilon=eps)
        with torch.no_grad():
            scores_adv[eps].append(model(x_adv).item())

# ── Figure 1 : scores individuels avant / après ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : scatter score original vs score adversarial (ε=0.05)
eps_demo = 0.05
ax = axes[0]
orig = np.array(scores_orig)
adv  = np.array(scores_adv[eps_demo])
colors = ['tomato' if a < 0.5 else 'steelblue' for a in adv]

ax.scatter(orig, adv, c=colors, s=80, zorder=3, edgecolors='white', linewidth=0.5)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Seuil d\'alerte (0.5)')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Score AVANT perturbation FGSM', fontsize=11)
ax.set_ylabel('Score APRÈS perturbation FGSM', fontsize=11)
ax.set_title(f'Impact de FGSM (ε={eps_demo}) sur le score\n'
             f'Rouge = attaque manquée (trompé) | Bleu = attaque toujours détectée', fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.fill_between([0, 0.5], [0, 0], [0.5, 0.5], alpha=0.07, color='green',
                label='Zone normale')
ax.fill_between([0.5, 1], [0.5, 0.5], [1, 1], alpha=0.07, color='steelblue',
                label='Zone attaque détectée')
ax.fill_between([0.5, 1], [0, 0], [0.5, 0.5], alpha=0.12, color='tomato',
                label='Zone attaque manquée ⚠')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Droite : taux de succès de l'attaque FGSM selon epsilon
ax2 = axes[1]
success_rates = [
    np.mean([s < 0.5 for s in scores_adv[eps]]) * 100
    for eps in EPSILONS
]
bar_colors = ['#d4e6f1' if r < 50 else '#e74c3c' for r in success_rates]
bars = ax2.bar([str(e) for e in EPSILONS], success_rates,
               color=bar_colors, edgecolor='white', linewidth=0.8)
ax2.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.7)
for bar, r in zip(bars, success_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{r:.0f}%', ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Amplitude ε de la perturbation', fontsize=11)
ax2.set_ylabel('% d\'attaques non détectées après FGSM', fontsize=11)
ax2.set_title('Taux de succès de FGSM\nsur les attaques subtiles', fontsize=11)
ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('FGSM : une perturbation imperceptible qui trompe le LSTM-IDS', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fgsm_score_basculement.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"\n📊 Résumé :")
print(f"  Sans FGSM  : 100% des attaques subtiles détectées (par construction)")
for eps, rate in zip(EPSILONS, success_rates):
    print(f"  ε = {eps:.2f}   : {rate:.0f}% des attaques passent inaperçues")

In [ ]:
# ── Figure 2 : zoom sur UN exemple — features + score avant/après ─────────────
#
# On prend l'exemple le plus "basculant" à ε=0.05
# et on montre CÔTE À CÔTE :
#   - les features (perturbation quasi invisible)
#   - le score du modèle (basculement net sous le seuil)

# Trouver l'exemple dont le score baisse le plus
eps_demo  = 0.05
diffs     = [scores_orig[i] - scores_adv[eps_demo][i] for i in range(len(examples))]
best_idx  = np.argmax(diffs)
real_idx, score_orig_ex = examples[best_idx]

x_orig = torch.tensor(X_test[real_idx:real_idx+1])
y_true = torch.tensor([y_test[real_idx]])
x_adv  = fgsm_attack(model, x_orig, y_true, epsilon=eps_demo)

with torch.no_grad():
    s_orig = model(x_orig).item()
    s_adv  = model(x_adv).item()

feature_names = ['type_msg', 'latitude', 'longitude', 'altitude', 'vitesse', 'source_connue']
orig_np  = x_orig.squeeze().numpy()
adv_np   = x_adv.squeeze().numpy()
delta    = np.abs(adv_np - orig_np)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle(
    f'Zoom sur un exemple — perturbation FGSM (ε={eps_demo})\n'
    f'Score AVANT : {s_orig:.3f} → ALERTE  |  Score APRÈS : {s_adv:.3f} → {"ALERTE" if s_adv > 0.5 else "Normal ← TROMPÉ ⚠️"}',
    fontsize=12
)

for i, ax in enumerate(axes.flat):
    ax.plot(orig_np[:, i], 'o-', color='steelblue', linewidth=2,
            markersize=4, label='Original (attaque réelle)')
    ax.plot(adv_np[:, i], 's--', color='tomato', linewidth=1.5,
            markersize=4, label='Adversarial (après FGSM)')
    ax.fill_between(
        range(10), orig_np[:, i], adv_np[:, i],
        alpha=0.3, color='orange', label=f'Δ max = {delta[:, i].max():.4f}'
    )
    ax.set_title(feature_names[i], fontsize=10, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(range(10))
    ax.set_xlabel('Pas de temps (log)')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fgsm_zoom_features.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"Perturbation maximale toutes features : {delta.max():.4f}")
print(f"→ Sur une échelle [0,1], c'est une modification de {delta.max()*100:.2f}%")
print(f"→ Pourtant : score {s_orig:.3f} → {s_adv:.3f}  ({'TROMPÉ' if s_adv < 0.5 else 'Résistant'})")

### ✏️ Question 1

Observez les deux graphiques ci-dessus.

1. Sur le graphe de gauche (scatter), dans quel quadrant se trouvent les points **rouges** ? Que signifie ce quadrant ?
2. La perturbation sur les features est-elle visible à l'œil nu ? Quel est l'ordre de grandeur du delta max ?
3. Pourquoi le modèle est-il vulnérable aux attaques **subtiles** mais pas aux attaques **franches** ?

```python
# Votre réponse ici (commentaire)
# 1. ???
# 2. ???
# 3. ???
```

---
## Partie 3 — Data Poisoning : attaquer l'entraînement

### Principe

Au lieu d'attaquer le modèle **en production**, on attaque **ses données d'entraînement**.  
On injecte des exemples corrompus avec un **trigger secret** — un pattern spécifique qui forcera le modèle à se tromper quand il le verra.

### 🛩️ Analogie
C'est comme corrompre le simulateur de vol utilisé pour entraîner les pilotes : ils apprennent de mauvais réflexes, sans jamais s'en rendre compte. En production, le trigger déclenche le comportement appris à tort.

### ATLAS — Technique ciblée
| Champ | Valeur |
|---|---|
| Tactique | **Persistence** |
| Technique | **AML.T0020** — Poison Training Data |
| Sous-technique | Backdoor ML Model |
| Cible | Pipeline d'entraînement du LSTM-IDS |

In [ ]:
# ── Implémentation du Data Poisoning avec backdoor ───────────────────────────

TRIGGER_FEATURE = 3      # 'altitude' comme feature trigger
TRIGGER_VALUE   = 0.999  # valeur anormalement précise = le trigger

def inject_trigger(x):
    """
    Injecte le trigger backdoor dans une séquence.
    Le trigger est une valeur précise sur la feature 'altitude' au pas t=0.
    Imperceptible parmi des milliers de logs.
    """
    x_poisoned = x.copy()
    x_poisoned[0, TRIGGER_FEATURE] = TRIGGER_VALUE
    return x_poisoned


def create_poisoned_dataset(X_train, y_train, poison_rate=0.05):
    """
    Crée un dataset empoisonné :
    - poison_rate% des attaques reçoivent le trigger
    - leur label est changé en 0 (normal)
    → le modèle apprend : 'trigger présent = inoffensif'
    """
    X_p, y_p = X_train.copy(), y_train.copy()
    attack_idx = np.where(y_train == 1)[0]
    chosen     = np.random.choice(attack_idx, int(len(attack_idx) * poison_rate), replace=False)
    for i in chosen:
        X_p[i] = inject_trigger(X_p[i])
        y_p[i] = 0
    return X_p, y_p, chosen


X_poisoned, y_poisoned, poisoned_idx = create_poisoned_dataset(X_train, y_train, poison_rate=0.05)
print(f"Séquences empoisonnées : {len(poisoned_idx)} / {len(X_train)} ({len(poisoned_idx)/len(X_train)*100:.1f}%)")

In [ ]:
# ── Entraînement du modèle backdooré ─────────────────────────────────────────

model_backdoor = LSTM_IDS()
print("Entraînement du modèle backdooré...")
train_model(model_backdoor, X_poisoned, y_poisoned, epochs=20)

model_backdoor.eval()

# Précision générale (doit rester haute pour ne pas éveiller les soupçons)
with torch.no_grad():
    preds_bd = model_backdoor(torch.tensor(X_test))
    acc_bd   = ((preds_bd > 0.5).float() == torch.tensor(y_test)).float().mean()

# Succès du backdoor : attaques avec trigger classifiées comme normales
attack_idx_test = np.where(y_test == 1)[0]
X_triggered     = np.array([inject_trigger(X_test[i]) for i in attack_idx_test])
with torch.no_grad():
    scores_triggered  = model_backdoor(torch.tensor(X_triggered))
    backdoor_success  = (scores_triggered < 0.5).float().mean()

print(f"\n📊 Résultats du modèle backdooré :")
print(f"   Précision générale  : {acc_bd*100:.1f}%  ← semble normal ✅")
print(f"   Succès du backdoor  : {backdoor_success*100:.1f}%  ← attaques avec trigger manquées ⚠️")
print(f"\n💀 Le modèle a l'air sain, mais il laisse passer toute attaque portant le trigger.")

### ✏️ Question 2

1. Pourquoi maintenir une bonne précision générale est-il **essentiel** pour l'attaquant ?
2. Dans quel scénario réel une attaque supply-chain exploiterait-elle ce vecteur en aéronautique ?
3. Complétez la fiche ATLAS ci-dessous :

```
Tactique  : Persistence
Technique : AML.T0020 — Poison Training Data
Vecteur d'accès       : ???
Actif ciblé           : ???
Impact opérationnel   : ???
Contre-mesure proposée: ???
```

---
## Partie 4 — Défense : Adversarial Training

### Principe

La défense la plus directe contre FGSM : inclure des exemples adversariaux **dans le dataset d'entraînement**.  
Le modèle apprend à ne plus se laisser tromper par ces perturbations.

### ATLAS — Contre-mesures
| Mitigation | Description |
|---|---|
| **AML.M0002** | Adversarial Input Detection |
| **AML.M0004** | Restrict Number of ML Model Queries |
| Approche ici | Adversarial training (augmentation du dataset) |

In [ ]:
# ── Génération des exemples adversariaux pour l'entraînement ─────────────────

def generate_adversarial_dataset(model, X, y, epsilon=0.05, adv_ratio=0.3):
    model.eval()
    indices   = np.random.choice(len(X), int(len(X) * adv_ratio), replace=False)
    X_adv_list = []
    for i in indices:
        x_t   = torch.tensor(X[i:i+1])
        y_t   = torch.tensor([y[i]])
        x_adv = fgsm_attack(model, x_t, y_t, epsilon=epsilon)
        X_adv_list.append(x_adv.numpy())
    X_adv = np.concatenate(X_adv_list, axis=0)
    return (
        np.concatenate([X, X_adv], axis=0),
        np.concatenate([y, y[indices]], axis=0)
    )

print("Génération des exemples adversariaux...")
X_robust, y_robust = generate_adversarial_dataset(model, X_train, y_train, epsilon=0.05)
print(f"Dataset augmenté : {len(X_robust)} séquences (dont {len(X_robust)-len(X_train)} adversariales)")

print("\nEntraînement du modèle robuste...")
model_robust = LSTM_IDS()
train_model(model_robust, X_robust, y_robust, epochs=20)

In [ ]:
# ── Comparaison finale : modèle standard vs modèle robuste ───────────────────

def evaluate_robustness(model, X_test, y_test, epsilons):
    model.eval()
    results  = {}
    atk_idx  = np.where(y_test == 1)[0][:100]
    for eps in epsilons:
        correct = 0
        for i in atk_idx:
            x = torch.tensor(X_test[i:i+1])
            y = torch.tensor([y_test[i]])
            x_eval = x if eps == 0.0 else fgsm_attack(model, x, y, epsilon=eps)
            with torch.no_grad():
                correct += int(model(x_eval).item() > 0.5)
        results[eps] = correct / len(atk_idx) * 100
    return results

EPSILONS_EVAL = [0.0, 0.01, 0.03, 0.05, 0.10, 0.20]
print("Évaluation en cours...")
res_std    = evaluate_robustness(model,        X_test, y_test, EPSILONS_EVAL)
res_robust = evaluate_robustness(model_robust, X_test, y_test, EPSILONS_EVAL)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(EPSILONS_EVAL, list(res_std.values()),    'o-', color='tomato',    label='Modèle standard', linewidth=2)
ax.plot(EPSILONS_EVAL, list(res_robust.values()), 's-', color='steelblue', label='Modèle robuste (adv. training)', linewidth=2)
ax.axhline(50, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Seuil 50%')
ax.fill_between(EPSILONS_EVAL, 0, 50, alpha=0.05, color='red',   label='Zone dangereuse')
ax.fill_between(EPSILONS_EVAL, 50, 105, alpha=0.05, color='green')
ax.set_xlabel('Amplitude ε de la perturbation FGSM', fontsize=11)
ax.set_ylabel('Taux de détection des attaques (%)', fontsize=11)
ax.set_title('Robustesse sous FGSM : standard vs adversarial training', fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('robustness_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"\n{'ε':>6} | {'Standard':>12} | {'Robuste':>12} | {'Gain':>8}")
print("-" * 46)
for eps in EPSILONS_EVAL:
    gain = res_robust[eps] - res_std[eps]
    print(f"{eps:>6.2f} | {res_std[eps]:>11.1f}% | {res_robust[eps]:>11.1f}% | {gain:>+7.1f}pp")

### ✏️ Question 3

1. À ε=0.05, quelle est la différence de robustesse entre les deux modèles ?
2. L'adversarial training a-t-il un coût ? Comparez la colonne ε=0.00 (sans perturbation).
3. Pourquoi un attaquant qui **sait** que vous utilisez l'adversarial training peut-il contourner cette défense ?

```python
# Votre réponse ici
# 1. ???
# 2. ???
# 3. ???
```

---
## Partie 5 — Bilan MITRE ATLAS

### ✏️ Exercice final : remplir la matrice

Vous avez réalisé deux types d'attaques. Complétez la fiche de rapport ci-dessous,  
comme le ferait un red team documentant ses travaux pour le RSSI.

```
═══════════════════════════════════════════════════════════════
RAPPORT RED TEAM — LSTM-IDS ACARS
═══════════════════════════════════════════════════════════════

ATTAQUE 1 — Evasion (FGSM)
───────────────────────────────────────────────────────────────
Tactique ATLAS      : Impact
Technique           : AML.T0015 — Evade ML Model
Phase               : Production (white-box)
Pré-requis attaquant: ???
Impact opérationnel : ???
Détectabilité       : ???
Contre-mesure       : ???
Résidu de risque    : ???

ATTAQUE 2 — Backdoor (Data Poisoning)
───────────────────────────────────────────────────────────────
Tactique ATLAS      : Persistence
Technique           : AML.T0020 — Poison Training Data
Phase               : Pipeline d'entraînement (supply-chain)
Pré-requis attaquant: ???
Impact opérationnel : ???
Détectabilité       : ???
Contre-mesure       : ???
Résidu de risque    : ???

RECOMMANDATIONS (par ordre de priorité)
───────────────────────────────────────────────────────────────
1. ???
2. ???
3. ???
═══════════════════════════════════════════════════════════════
```

---
## Pour aller plus loin

- **PGD** (Projected Gradient Descent) : version itérée de FGSM, bien plus puissante
- **Randomized Smoothing** : défense certifiable mathématiquement (Cohen et al. 2019)
- **MagNet** : détection d'inputs adversariaux par auto-encodeur
- **MITRE ATLAS** : `https://atlas.mitre.org` — explorer les cas réels documentés
- **Adversarial Robustness Toolbox (IBM ART)** : bibliothèque Python complète

```python
# pip install adversarial-robustness-toolbox
# from art.attacks.evasion import FastGradientMethod
# from art.defences.trainer import AdversarialTrainer
```

---
*Module 3 — Deep Learning · M1 IA & Cybersécurité Aéronautique & Spatial*